# Caso: Modelo de Predicción Climática en Australia

## Objetivo
El objetivo de este caso  es construir un modelo que permita predecir si mañana va a llover (o no) en Australia en función a mediciones climáticas históricas.

Recursos Adicionales

- [scikit-learn documentación de árboles de decisión](http://scikit-learn.org/stable/modules/tree.html)
- [Gini Vs Entropia](http://www.garysieling.com/blog/sklearn-gini-vs-entropy-criteria)

## 1. Análisis descriptivo del dataset

El set de entrenamiento a utilizar es **weatheraus_entrenamiento.csv** el cual contiene información sobre 51.785 mediciones climáticas realizadas en distintos lugares de Australia. En el archivo **Diccionario-Wheather-AUS** se especifica la descripción de cada uno de sus atributos y algunas aclaraciones sobre la información suministrada.

La variable target es **RainTomorrow**, que especifica si llueve o no el día siguiente.

Descripción más detallada del dataset:
  http://www.bom.gov.au/climate/dwo/IDCJDW0000.shtml

In [ ]:
# Librerias a importar
import numpy as np
import pandas as pd
import pydotplus
import seaborn as sns
import copy
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Por defecto, matplotlib crea una figura en una ventana separada. Cuando usamos notebooks, podemos hacer que las
# figuras aparezcan en línea dentro del notebook. Esto lo hacemos ejecutando:
%matplotlib inline

from IPython.display import Image

# Scikit-learn (sklearn) es una librería que implementa algunos algoritmos de Machine Learning y pre-procesamiento de datos.
from sklearn.tree import export_graphviz

from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split

from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import make_scorer, accuracy_score
from sklearn import metrics

from sklearn.svm import SVR
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder


In [ ]:
# Leemos el dataset de entrenamiento
dataset_entrenamiento='https://raw.githubusercontent.com/unlam-fcdin/UNLaM_FCDIN/master/weather_aus_entrenamiento.csv'
df = pd.read_csv(dataset_entrenamiento, sep =',', na_values = '.', parse_dates=["Date"])

# Para estandarizar el análisis, llamaremos variable "CLASE" a la variable target
df['CLASE'] = df['RainTomorrow']
df.drop(["RainTomorrow"], axis=1, inplace=True)

df.head(5)

#### **Descripción Estadística General**
- **Consigna**: ¿Qué información relevante obtenemos de este primer análisis?

In [ ]:
# ¿Qué información estadística obtenemos del dataset?
df.describe()

In [ ]:
# Veamos la distribución de la variable target
print(df['CLASE'].value_counts(normalize = True)*100) # expresado en porcentajes
print(df['CLASE'].value_counts(normalize = False)) # expresado en nominal

sns.countplot(x='CLASE', data=df,  palette="cubehelix")

### 1.a) Análisis de Missing Values o Nulos

Analizamos la existencia de valores faltantes y seleccionamos la estrategia de imputación.

In [ ]:
val_nulos = df.isnull().sum()
print(df.shape)
print(val_nulos)

#### **Imputación de Nulos**
- **Consigna**: ¿Qué criterios de imputación aplicaría y en qué variables?

Cabe resaltar que la imputación debe realizarse en la fase de **Preparación de Datos**.

In [ ]:
# Graficamos la distribución de valores nulos en cada variable
atributos_nulos = (df.isnull()).sum(axis=0)
atributos_nulos = pd.DataFrame(atributos_nulos, columns=['Cantidad de Nulos'])
atributos_nulos = atributos_nulos.sort_values(by=['Cantidad de Nulos'], ascending=True)
atributos_nulos.drop(['CLASE'], inplace = True)
atributos_nulos.plot(kind='barh', figsize=(15,5), color='orange', grid=False)

### 1.b) Análisis de Outliers

Analizamos los valores outliers de cada variable y determinamos si queremos/debemos imputarlos o no.

In [ ]:
# Creamos un gráfico boxplot para analizar las variables
for c in df.columns:
  if("float" in str(df.dtypes[c]) or "int" in str(df.dtypes[c])):
    plt.figure(figsize=(10, 3))
    sns.boxplot(x=c, data=df, orient = 'h', palette="Set2")

#### **Imputación de Outliers**

**Consignas**
- ¿En qué variables realizaría imputación de outliers y qué criterio aplicaría (1,5 desvios, 2 desvios, 3 desvios...) ?

- ¿Alguna variable requiere especial atención? ¿Cúal/es?

Cabe resaltar que la imputación debe realizarse en la fase de **Preparación de Datos**.

### 1.c) Análisis de Correlación de Variables Numéricas

Una matriz de correlación permite estudiar la relación lineal o comportamiento que puede existir entre dos o más variables.

  - Correlación positiva: ocurre cuando una variable aumenta y la otra también.
  - Correlación negativa: es cuando una variable aumenta y la otra disminuye.
  - Sin correlación: no hay una relación aparente entre las variables.

In [ ]:
# Creamos una variable numérica CLASE_NUM a partir de nuestra variable objetivo CLASE.
# La misma tendrá un valor 1 cuando llovió el dia siguiente y un valor 0 en caso contrario
# Esta variable se usará para calcular la correlacion lineal entre las variables númericas y la variable objetivo.
df['CLASE_NUM'] = list(map(lambda clase: 1 if (clase == 'Yes') else 0, df['CLASE']))
df_types = df.dtypes.reset_index()
ordinals_cols = list(df_types[(df_types[0]=="float64") | (df_types[0]=="int64")]["index"])

# Calculamos la matriz de correlacion entre todas las variables del dataset.
df[ordinals_cols].corr()

In [ ]:
# Seleccionamos sólo la correlación de la variable objetivo numérica CLASE_NUM.
dfd = df[ordinals_cols].corr()[["CLASE_NUM"]]*100
dfd

In [ ]:
# Borramos la correlación de la variable objetivo numérica consigo misma.
dfd = dfd.drop("CLASE_NUM", axis=0)   # CLASE_NUM = 100

# Ordenamos las variables de forma decreciente por el valor de correlación positiva con la variable objetivo.
dfd = dfd.sort_values(["CLASE_NUM"], ascending=False)
dfd

#### **Correlación de Variables**

**Consignas**
- ¿Qué variables son interesantes para la predicción a realizar?

- ¿Qué acciones realizaría con ellas para aumentar el poder predictivo del modelo a generar?

Cabe resaltar que cualquier actividad a realizar sobre el set de datos, debe realizarse en la fase de **Preparación de Datos**.

In [ ]:
# Graficamos el mapa de calor.
plt.figure(figsize=(10, 20))
sns.heatmap(dfd, robust=True, linewidths=.5, annot=True, )

df.drop(["CLASE_NUM"], axis=1, inplace=True)

### 1.d) Análisis del nivel de humedad

Vamos a analizar ahora cómo se relaciona el nivel de humedad (Humidity3pm) con la probabilidad de que mañana llueva.

In [ ]:
# Creamos un gráfico boxplot para analizar el nivel de humedad respecto a la CLASE
plt.figure(figsize=(4, 8))
s=sns.boxplot(x="CLASE", y="Humidity3pm", data=df,  palette="cubehelix")
s.plot()

#### **Distribución de variables**
- **Consigna**: ¿Se observa alguna relación respecto a la humedad entre los días que llovió y los que no?

Realizar el mismo análisis para todas las variables númericas y analizar su relación con el target de predicción.


### 1.e) Análisis de las variables categóricas de forma visual
En este punto vemos cómo comparar la distribución de una variable categórica (por ejemplo RainToday) vs nuestro target de predicción. Este mismo análisis debería realizarse para todas las variables categóricas del set de datos.

In [ ]:
# Distribución de la probabilidad de que llueva mañana, dado que llovió hoy (variable RainToday)

# Copiamos el dataset para agregar una 3ra categoría para los nulos
df2 = copy.copy(df)
df2["RainToday"] = df2["RainToday"].fillna('Null')

ct = pd.crosstab(df2.RainToday, df.CLASE, margins=False)   # armamos una tabla cruzada
for c in df2.RainToday.unique():
  ct.loc[[c]].plot.barh(stacked=True, color=sns.color_palette("cubehelix", len(df2.RainToday.unique())))  # loc trabaja en las etiquetas del índice para armar un gráfico de barras)

# Borramos el dataset creado para imputar los nulls en la 3ra categoria
del df2

plt.show()

#### **Correlación en Variables Categóricas**
**Consignas**
- ¿Se observa alguna relación entre la probabilidad de que llueva mañana en función de si hoy llovió?

- ¿Qué puede decir sobre la categoría Null? ¿Que acciones tomaría?


Cabe resaltar que cualquier actividad a realizar sobre el set de datos, debe realizarse en la fase de **Preparación de Datos**.

## 2. Preparación de datos

Una vez cargados los datos y analizados, se deben preparar para ser procesados. Los 3 pasos generales que se deben seguir son:

1. Tratamiento de outliers
2. Feature engineering
3. Tratamiento de missing values (o valores nulos)

Los pasos detallados de la fase, son:

- Leer los datos con Pandas.
- Comprobar si hay valores nulos y crear todas las variables nuevas.
- Encodear todos los atributos categóricos como booleanos usando `pd.get_dummies`
- Encodear las etiquetas usando `LabelEncoder`
- Construir una variable Y que contenga la variable objetivo a predecir y una vector X que contenga todo el resto de variables a usar en la predicción.
- Dividir el dataset completo en training y testing
- Dividir X e y con train_test_split así:
        train_test_split(X, y, test_size=0.3, random_state=42)

Para el paso de feature engineering se podría usar la libreria "featuretools" la documentación se puede encontrar en:
https://jakevdp.github.io/PythonDataScienceHandbook/05.04-feature-engineering.html

Con esta librería se generarán automáticamente las combinaciones "primitivas" posibles entre las variables que tenemos. Como ejemplo, la división de una variable con otra, el máximo, minimo, mediana, etc.

Vamos a crear una función llamada **preparacion_de_datos(dataset)** que incluirá todas las tareas de preparación de datos necesarias para construir nuestro modelo predictivo. Esta función la vamos a utilizar tanto para el dataset de entrenamiento como para el de prueba.

****A partir de este momento, la variable CLASE (objetivo) deja de ser una etiqueta para convertirse en una variable numérica, con valores: 0 (NO LLUEVE) 1 (LLUEVE)****

In [ ]:
# FUNCION preparacion_de_datos (:parametros)
#   df_e            => Dataset de entrada a modificar
#   imputar_ouliers => Flag que indica si se deben imputar outliers o no
#   imputar_nulos   => Flag que indica si se deben imputar nulos o no

def preparacion_de_datos(df_e, imputar_ouliers, imputar_nulos):
  # Comenzamos haciendo una copia del dataset que la función recibe como parámetro de entrada
  df_s = copy.copy(df_e)

  # Quitamos el atributo Date, aunque se podría jugar con esta variable
  df_s  = df_s.drop(['Date'], axis=1)

  # (*1) ---- IMPUTACIÓN DE OUTLIERS ----

  # En este punto se deben imputar los valores outliers, sólo si se indicó por parámetro(imputar_outliers)
  if imputar_ouliers:
    print("TODO: Imputación de outliers.")

    # Como ejemplo imputamos la variable "Evaporation" que vimos necesitaba imputación sólo de outliers superiores

    # Primero definimos una función para calcular la media, con los parámetros:
    ## dff -> Dataframe a usar
    ## c   -> Columna a imputar
    ## min -> Límite inferior, si no aplica no se envía.
    ## max -> Límite superior, si no aplica no se envía.

    def calcular_media(dff, c, min=None, max=None):
      # Seteamos el mínimo y máximo, si viene por parámetro. Si no fueron informados, tomamos el actual del set de datos.
      minimo = dff[c].min() if min==None else min
      maximo = dff[c].max() if max==None else max

      # Filtramos los registros dentro del rango
      dff2 = dff[(dff[c]>=minimo) & (dff[c]<=maximo)]

      # Devolvemos la media
      return dff2[c].mean()

    # Calculamos el límite superior y la media sin los outliers para luego imputar.
    # Lo hacemos por fuera para no ejecutar el cálculo una vez por cada registro
    outlier_superior = df_s['Evaporation'].mean() + 1.5*df_s['Evaporation'].std()
    media_sin_outliers = calcular_media(df_s, 'Evaporation', max=outlier_superior)

    # Imputamos por la media los outliers superiores
    df_s['Evaporation'] = df_s.apply(lambda x: media_sin_outliers if x['Evaporation']>outlier_superior else x['Evaporation'], axis=1)


  # (*2) ---- FEATURE ENGINEERING ----

  # Cada unx puede crear los atributos que considere necesario (y mejoren la predicción)
  print("TODO: Creación de nuevas variables.")

  # Dejamos un ejemplo de como crear un atributo, hay que dar especial atención a los atributos
  # que son del tipo cociente, ya que agregan información que los métodos de clasificación
  # en general no pueden obtener:

  ## 1) Cociente entre la nubosidad a las 9am y a las 3pm para ver la variación de nubosidad
  ## Las variables "Cocientes" las vamos a indicar con "c_")
  df_s['c_Cloud_3vs9'] = list(map(lambda cloud9, cloud3:
                                      round( ( (cloud3 or 0)/cloud9 if ( (cloud9 or 0) != 0 ) else 0 ), 2) ,
                                      df_s['Cloud9am'],
                                      df_s['Cloud3pm']))

  ## 1) Cociente entre la humedad a las 9am y a las 3pm para ver la variación de nubosidad
  ## Las variables "Cocientes" las vamos a indicar con "c_")
  df_s['c_Humidity_3vs9'] = list(map(lambda Humidity9, Humidity3:
                                      round( ( (Humidity3 or 0)/Humidity9 if ( (Humidity9 or 0) != 0 ) else 0 ), 2) ,
                                      df_s['Humidity9am'],
                                      df_s['Humidity3pm']))


  # (*3) ---- TRATAMIENTO DE VALORES NULOS ----

  # Veamos si tenemos valores nulos o infinitos. Como ejemplo, podemos optar por setear el valor 0 por defecto.
  if imputar_nulos:
    print("TODO: Imputación de valores nulos.")

    # Posible imputación de nulos
    # Imputamos por la media a los nulos de las cuatro variables con muchisimos nulos
    #for c in ['Evaporation','Sunshine','Cloud9am','Cloud3pm']:
      #media = calcular_media(df_s, c)
      #df_s[c] = df_s.apply(lambda x: media if x[c] is None else x[c], axis=1)

    # A la variable RainToday, le relleno los nulos con una nueva categoria Null
    df_s["RainToday"] = df_s["RainToday"].fillna('Null')

    # Al resto los fuerzo 0
    df_s[df_s==np.inf]=np.nan
    df_s.fillna(0, inplace=True)


  # Tratamiento especial variables categóricas con alta dimensionalidad, ej:"Location"
  # Creamos un índice de humedad de las 3pm por location
  df_s['Location_Humidity3pm'] = df_s.groupby('Location')['Humidity3pm'].transform('mean')
  df_s['c_Location_Humidity3pm'] = list(map(lambda humidity, location_himdity:
                                            round( ( (humidity or 0)/location_himdity if ( (location_himdity or 0) != 0 ) else 0 ), 2) ,
                                            df_s['Humidity3pm'],
                                            df_s['Location_Humidity3pm']))

  # Creamos un índice de %días que llovió historicamente por location
  # En este caso lo estamos haciendo transversal, lo ideal sería hacerlo relativo a las fechas previas a la acual.
  df_s['RainTodayN'] = df_s.apply(lambda x: 1 if x['RainToday']=='Yes' else 0, axis=1)
  df_s['idd'] = 1
  df_s['Location_RainTodayN_Yes'] = df_s.groupby('Location')['RainTodayN'].transform('sum')
  df_s['Location_RainTodayN_Tot'] = df_s.groupby('Location')['idd'].transform('sum')
  df_s['c_Location_RainToday_Yes'] = list(map(lambda rain_today, total:
                                            round( ( (rain_today or 0)/total if ( (total or 0) != 0 ) else 0 ), 3) ,
                                            df_s['Location_RainTodayN_Yes'],
                                            df_s['Location_RainTodayN_Tot']))

  # Borramos las variables que ya no nos sirven
  df_s.drop(["Location", "RainTodayN","idd","Location_RainTodayN_Yes","Location_RainTodayN_Tot"], axis=1, inplace=True)

  # Convertimos la variable clase en un numérico con 0 o 1
  df_s['CLASE'] = list(map(lambda clase: 1 if (clase == 'Yes') else 0, df_s['CLASE']))

  return df_s

In [ ]:
# Ejecutamos la función de preparacion de datos.
# Indicando que si queremos imputar ouliers y si queremos imputar nulos
# Este el primer paso que debemos realizar para tener todas las variables a utilizar.
df = preparacion_de_datos(df, True, True)

df.head()

In [ ]:
print(df.shape)

In [ ]:
# Dejamos en el dataset de entrenamiento todas las variables, excepto la CLASE.

# Encodeamos todos los atributos categóricos como booleanos con la función pd.get_dummies (sin incluir la variable objetivo).

X  = pd.get_dummies(df.drop(['CLASE'], axis=1))
atributos = X.columns

# Encodeamos las etiquetas usando LabelEncoder
# Convertimos la variable objetivo en una variable booleana de valores 0 o 1 para simplificar los cálculos

le = LabelEncoder()
y = le.fit_transform(df['CLASE'])


# Dividimos el dataset en entrenamiento y prueba (70% para training y 30% para testing)
# Dividimos X e y con la funcion train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y,
                                                    test_size=0.3,
                                                    random_state=42)

print(X_train.shape)
X_train.head(5)

#### **Preparación de Datos**
**Consignas**
- ¿Se observa algo objetable en el set de datos generado?

- ¿Hay alguna variable que requiera un tratamiento adicional?


Cabe resaltar que cualquier actividad a realizar sobre el set de datos, debe realizarse en la fase de **Preparación de Datos**.

## 3. Construcción del modelo predictivo sin optimizacion de hiperparámetros

### 3.a) Modelo inicial

Vamos a construir un árbol de clasificación inicial usando el dataset. Los pasos que realizaremos son:

- Ajustar un árbol de clasificación con `max_depth=3`
- Visualizar el árbol usando graphviz
- Calcular la importancia de los atributos
- Calcular y mostrar la matriz de confusión
- Sacar la restricción de `max_depth=3` y ver si la clasificación mejora

In [ ]:
# Ajustamos un árbol de clasificación con max_depth=3
treeclf = DecisionTreeClassifier(max_depth=3, random_state=1)
treeclf.fit(X_train, y_train)

### 3.b) Interpretación de resultados
- Visualizaremos el árbol de decision generado junto con las condiciones de split y los nodos resultantes.
- Analizaremos cómo contribuyeron las variables usadas en la predicción.
- Calcularemos el error de entrenamiento y de testeo del modelo generado.


In [ ]:
# Visualizamos el árbol de decisión usando graphviz
dot_data = export_graphviz(treeclf, out_file=None,
                feature_names=atributos,
                filled=True, rounded=True,
                special_characters=True)
graph = pydotplus.graph_from_dot_data(dot_data)
Image(graph.create_png())

### 3.b.i) Lectura del árbol de decisión generado

#### **Interpretación de Resultados**
- **Consigna**: Indicar, por lo menos, dos reglas de relevancia donde la predicción sea positiva (Mañana Llueve).

-	Humidity3pm > 71.5	->	Humidity3pm > 83.5	-> Temp9am > 7.95		=> P('LLUEVE') = 1740 / 2074 = 83.9%

-	Humidity3pm > 71.5	->	Humidity3pm <= 83.5	-> Windgustspeed > 47.0	=> P('LLUEVE') = 596  /  919 = 64.85%

### 3.b.ii) Curva ROC

La curva ROC nos permite visualizar cómo se distribuyen los casos positivos y negativos dentro de nuestro modelo.

En resumen, nos indica qué % de casos positivos (clase target) vamos a captar si tomamos X % de negativos, esto da una clara idea del poder predictivo del algoritmo. Este valor se resume en la métrica AUC (Area Under Curve), la cual indica qué tan "rápido" crece la curva (incluye mayor % de positivos) conforme avanza la cantidad de negativos.

Los **mejores algoritmos**, son aquellos que tienen curvas que **comienzan con pendientes muy grandes y crecen verticalmente muy rápido**, ya que esto indica que captan mayor cantidad de positivos para la misma cantidad de negativos.


In [ ]:
plt.figure(figsize=(15, 8))

y_pred_proba = treeclf.predict_proba(X_test)[::,1]
fpr, tpr, _ = metrics.roc_curve(y_test,  y_pred_proba)
auc = metrics.roc_auc_score(y_test, y_pred_proba)
print("AUC - Area Under the Curve - Área Bajo la Curva: {}".format(auc))

# Graficamos la curva roc del arbol
plt.plot(fpr,tpr,label="DT MAX_DEPTH:3 - AUC="+str(round(auc,3)))
plt.xlabel('% No llueve', fontsize=14)
plt.ylabel('% Llueve', fontsize=14)
plt.legend(loc=4, fontsize=12)

# Graficamos la recta del azar
it = [i/100 for i in range(100)]
plt.plot(it,it,label="AZAR, AUC=0.5",color="red")

plt.title("Curva ROC", fontsize=14)
plt.tick_params(labelsize=12);
plt.show()

Cuando se prueban distintos algoritmos, lo ideal es dibujar todas las curvas ROC en el mismo gráfico y comparar cómo se distancian unas de otras. En general, no existen grandes diferencias, pero son las pequeñas diferencias las que generan mayores ganancias.

Para comparar el poder predictivo de los algoritmos también se utiliza la métrica AUC (Área Bajo la Curva) pero esta métrica es genérica de la predicción completa y no aporta distintas visiones, depende del corte que querramos aplicar para saber qué efectividad tiene.


### 3.b.iii) Importancia de las variables

- La primera columna es el nombre de la variable y la segunda indica la importancia relativa a la predicción. Los valores más altos indican que esa variable tiene mayor importancia para "dividir" las distintas clases de la variable objetivo.

- Hay que tener claro que esto no implica una relación positiva, la importancia mide tanto relaciones positivas como las relaciones negativas (inversa).


#### **Importancia de las variables**
- **Consigna**: Indicar las 10 variables con mayor importancia en el modelo y su importancia relativa al mismo.

In [ ]:
pd.DataFrame({'Atributo':atributos,
              'importancia':treeclf.feature_importances_}).sort_values('importancia',
                                                                       ascending=False).head(10)

### 3.b.iv) Matriz de Confusión del Modelo y Errores de Predicción
Analizando la matriz de confusión, podemos entender en qué casos el modelo acierta en la predicción y cuáles falla. En base a esto, podemos determinar las métricas de evaluación de modelos.

#### **Evaluación de Modelos**
**Consignas**
- Calcular la matríz de confusión.
- Indicar las métricas principales: Exactitud (accuracy) - Error de Predicción - Precisión (precision) - Sensibilidad (recall)


In [ ]:
# Calculamos y mostramos la matriz de confusión del modelo
y_pred = treeclf.predict(X_test)
conf = confusion_matrix(y_test, y_pred)

predicted_cols = ['pred_'+str(c) for c in le.classes_]
pd.DataFrame(conf, index = le.classes_, columns = predicted_cols)


In [ ]:
# Reporte del clasificador
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

## 4. Función de Ganancia y Probabilidad de Corte

Vamos a definir una función de ganancia "teórica" con la probabilidad de corte para este caso práctico.

Esta función de ganancia consiste en **sumar 4 puntos por cada acierto** y **restar 1 punto por cada fallo**, para definir una métrica que podamos usar para la optimización del modelo. Estos puntajes nos definen una **probabilidad de corte** que determinan cuáles de las observaciones serán consideradas como positivas en el hecho que va a llover.

La misma puede calcularse partiendo de la función de ganancia:

- universo_total = dias('LLUEVE') + dias('NO LLUEVE')

- ganancia = **4** * dias('LLUEVE') - **1** * dias('NO LLUEVE')

Si descomponemos el 4 como la resta (5-1), también se podría escribir como:

- ganancia = **5** * dias('LLUEVE') - **1** * dias('LLUEVE') - **1** * dias('NO LLUEVE')

**( * )** Cuando se haga referencia a **dias('LLUEVE')** se refiere a la cantidad (#) de observaciones cuya clase es Yes (llovió al día siguiente), analogamente aplica para **dias('NO LLUEVE')**.
-

Si trabajamos un poco esta función de ganancia, buscando que la ganancia generada sea **mayor a 0**, podemos calcular la **probabilidad de corte**, si tenemos que ganancia > 0, entonces:
- 5 * dias('LLUEVE') - 1 * dias('LLUEVE') - 1 * dias('NO LLUEVE')  **>** 0
- 5 * dias('LLUEVE') - 1 * ( dias('LLUEVE') + dias('NO LLUEVE') )  **>** 0
- 5 * dias('LLUEVE') **>** 1 * ( dias('LLUEVE') + dias('NO LLUEVE') ))

- ( dias('LLUEVE') / ( dias('LLUEVE') + dias('NO LLUEVE') ) ) **>** 1 / 5

- **PROB('LLUEVE') > 0.2**

Esta función nos indica que la **probabilidad de corte es 0.2** aunque uno puede modificar ese valor para mejorar el modelo.

Este es un caso de uso de custom functions para scoring en Grid Searching, ya que por defecto la métrica usada para este tipo de algoritmos es AUC (Area Under the Curve) pero que en la vida real tiene poca aplicación.

In [ ]:
prob_corte = float(0.2) # probabilidad de corte calculada

#Definimos la función de ganancia a utilizar para el scoring en la búsqueda:
def funcion_ganancia(clf, X, y_true):
    y_prob_llueve = clf.predict_proba(X)[:, 1]

    ganancia = sum([(4 if y_true[i] > 0 else -1)
                    if y_prob_llueve[i]  > prob_corte
                    else 0
                    for i in range(len(y_prob_llueve))])

    return ganancia

## 5. Construcción del modelo usando GridSearchCV con optimización de hiperparámetros

Las clase **GridSearchCV** se utiliza para automatizar la selección de los parámetros de un modelo, aplicando para ello la técnica de validación cruzada. Partiendo de un modelo y un conjunto de parámetros, GridSearchCV prueba múltiples combinaciones y selecciona los valores de los parámetros que ofrecen mayor rendimiento para un modelo y conjunto de datos.

Los parámetros a optimizar son:

Medida            | Que hace
------------------|-------------
max_depth         | limita la altura del árbol (niveles-2)
max_features      | limita la cantidad de atributos a utilizar en una división
max_leaf_nodes    | limita la cantidad máxima de nodos hoja puede tener el árbol
min_samples_leaf  | cantidad mínima de muestras de una hoja
min_samples_split | cantidad mínima de muestras para dividir un nodo

Cada uno puede definir sus propios rangos de valores para cada parámetro, considerando que cuantos más parámetros se usen, el tiempo de procesamiento crece exponencialmente.

### 5.a) Definición de parámetros

In [ ]:
# Definimos los parametros a evaluar:

PARAMETROS = {'max_depth':[3, 10, 12, 15, 50],
              'max_features':[10, 25, 35, 40],
              'max_leaf_nodes':[10, 50, 12, 100, 1000],
              'min_samples_leaf':[20, 50, 100, 500],
              'min_samples_split':[50, 100, 200, 1000]}

k_n_jobs = 2 # numero de iteraciones definidas

# Hacemos la búsqueda con GridSearchCV

model = DecisionTreeClassifier(random_state=1)  # modelo de árbol de decision, podría ser cualquier otro o iterar una lista de modelos para probar y sus parametros
gs = GridSearchCV(model,
                  PARAMETROS,
                  n_jobs=k_n_jobs,
                  scoring=funcion_ganancia,
                  cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=1), #Cross Validation de 5 capas
                  verbose=1)
gs.fit(X_train, y_train)

# Mostramos los mejores resultados obtenidos

print(gs.best_estimator_)

In [ ]:
#print('Puntaje del modelo en CV: {:.2f}'.format(gs.best_score_))
print('Puntaje del modelo en Testing: {:.2f}'.format(funcion_ganancia(gs.best_estimator_, X_test, y_test)))

### 5.b) Interpretación de resultados
- Visualizaremos el árbol de decision generado con GridSearchCV  junto con las condiciones de split y los nodos resultantes.
- Analizaremos cómo contribuyeron las variables usadas en la predicción.
- Calcularemos el error de entrenamiento y de testeo del modelo generado.

In [ ]:
# Visualizamos el mejor árbol de decisión generado usando graphviz
dot_data=export_graphviz(gs.best_estimator_,
                         out_file=None,
                         feature_names=X.columns,
                         filled=True, rounded=True,
                         special_characters=True)
graph = pydotplus.graph_from_dot_data(dot_data)
Image(graph.create_png())

#### **Interpretación de Resultados**
- **Consigna**: Indicar, por lo menos, dos reglas relevantes donde la predicción sea positiva (Mañana Llueve).

### 5.b.i) Importancia de las variables

#### **Importancia de las variables**
**Consignas**
- Indicar las 10 variables con mayor importancia en el modelo y su importancia relativa al mismo.

- Analizar qué variables coinciden con el modelo previo y cuáles no.

In [ ]:
pd.DataFrame({'Atributo':atributos,
              'importancia':gs.best_estimator_.feature_importances_}).sort_values('importancia',
                                                                       ascending=False).head(15)

### 5.b.ii) Curva ROC
Comparamos las curvas ROC y métricas AUC de los distintos árboles generados por GridSearching, para entender qué valores de hiperparámetros mejoran la capacidad predictiva del algoritmo y qué tan grande (o pequeña) es la diferencia entre los árboles.

In [ ]:
# Como primer paso, vamos a probar todas las combinaciones de parámetros
# y vamos a guardar los resultados de cada árbol en la lista "resultados"
resultados = []

# Recorremos las combinaciones de parámetros y guardamos el árbol entrenado, parámetros y la predicción
for p in gs.cv_results_['params']:
  try:
    treeclf2 = DecisionTreeClassifier(max_depth= p['max_depth'],
                                      max_features= p['max_features'],
                                      max_leaf_nodes= p['max_leaf_nodes'],
                                      min_samples_split= p['min_samples_split'],
                                      random_state=1)
    treeclf2.fit(X_train, y_train)
    y_pred_proba = treeclf2.predict_proba(X_test)[::,1]

    # Guardamos el árbol, la predicción y los parámetros de cada ejecucion
    resultados.append({"árbol": copy.copy(treeclf2), "prediccion":copy.copy(y_pred_proba), "parámetros":copy.copy(p)})
  except Exception as e:
    print(e)

### Graficamos la curva ROC de cada árbol

In [ ]:
# Graficamos la curva ROC del arbol de cada iteracion
def graficarCurvaRoc(arbol):
  # Calculamos la probabilidad predicha
  y_pred_proba = r['prediccion']

  # Calculamos los valores de la curva ROC para graficar
  # VALIDAR AGREGAR EL PESO PARA CADA REGISTRO (POSITIVO O NEGATIVO) EN EL CALCULO DE LA ROCs
  fpr, tpr, _ = metrics.roc_curve(y_test,  y_pred_proba)

  # Calculamos el Area Bajo la Curva (AUC) y la guardamos
  auc = metrics.roc_auc_score(y_test, y_pred_proba)
  r['auc'] = auc

  # Graficamos
  plt.plot(fpr,tpr) #,label= "AUC="+str(auc))
  plt.legend(loc=4, fontsize=12)

# Inicializamos los labels del gráfico
plt.figure(figsize=(20, 10))
plt.xlabel('% No llueve', fontsize=14)
plt.ylabel('% Llueve', fontsize=14)

# Graficamos la recta del azar
it = [i/100 for i in range(100)]
plt.plot(it,it,label="AZAR, AUC=0.5",color="black")

# Para cada árbol probado (en la variable resultados) graficamos la curva ROC
for r in resultados:
    graficarCurvaRoc(r)

# Agregamos el titulo y configuro el tamaño de letra
plt.title("Curva ROC", fontsize=14)
plt.tick_params(labelsize=12);
plt.show()

#### **Curva ROC**
**Consignas**
- ¿Cuál es el mejor modelo?
- ¿Qué porcentaje de corte eligirían?


### 5.b.iv) Matriz de Confusión del Modelo y Errores de Predicción

#### **Evaluación de Modelos**
**Consignas**
- Calcular la matríz de confusión.
- Indicar las métricas principales: Exactitud (accuracy) - Error de Predicción - Precisión (precision) - Sensibilidad (recall)


In [ ]:
# Calculamos y mostramos la matriz de confusión del modelo
y_pred = gs.best_estimator_.predict(X_test)
conf = confusion_matrix(y_test, y_pred)

predicted_cols = ['pred_'+str(c) for c in le.classes_]
pd.DataFrame(conf, index = le.classes_, columns = predicted_cols)


In [ ]:
# Reporte del clasificador
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

## 6. Aplicando el modelo a datos nuevos

Ahora podemos aplicar el modelo generado a los datos nuevos donde no conocemos la clase para predecir el futuro. El set de prueba con los nuevos datos a utilizar es **weather_aus_predecir.csv**.


In [ ]:
# Leemos el dataset de aplicacion, para predecir la lluvia en casos que desconocemos el futuro.
dataset_aplicacion='https://raw.githubusercontent.com/unlam-fcdin/UNLaM_FCDIN/master/weather_aus_predecir.csv'
df_apply = pd.read_csv(dataset_aplicacion, sep =',', na_values = '.', parse_dates=["Date"])

# Para estandarizar el análisis, llamaremos variable "CLASE" a la variable target
df_apply['CLASE'] = df_apply['RainTomorrow']
df_apply.drop(["RainTomorrow"], axis=1, inplace=True)

# Siempre el primer paso es ejecutar la función de preparacion de datos que construimos
df_apply = preparacion_de_datos(df_apply, True, True)

# Borramos la CLASE y el campo FOTO_MES, ya que ambos son constantes, en este caso.
df_apply = df_apply.drop(['CLASE'], axis=1)

# Veamos como quedó el dataset
df_apply.head(10)


In [ ]:
# Encodeamos todos los atributos categóricos como booleanos con la función pd.get_dummies (sin incluir la variable objetivo).
X_apply = pd.get_dummies(df_apply)
atributos = X_apply.columns

# Aplicamos la predicción al nuevo dataset
scores = gs.best_estimator_.predict_proba(X_apply)

# Agregamos al dataset la probabilidad de lluvia predicha con el modelo
df_result = copy.copy(df_apply)
df_result['PROB'] = scores[:,1]

# Veamos como quedó:
df_result.sort_values(by='PROB', ascending=False).head(5)

## 7. Entregable Final
Para terminar, vamos a generar un archivo de salida "csv" con los casos donde la medición realizada infiera que el día de mañana va a llover en ese lugar. Esto lo realizamos en función de la probabilidad de corte para filtrar los casos con baja probabilidad.

In [ ]:
dataset_entrega='dataset_entrega.csv'

# Entregamos sólo las observaciones donde predecimos que va a llover
df_entregar = df_result[df_result.PROB > prob_corte]
df_entregar.to_csv(path_or_buf=dataset_entrega, sep=",", na_rep='.')
df_entregar.head(10)

In [ ]:
df_entregar[df_entregar.PROB < 0.22]